In [15]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path


# ============================================================
# 1. Load files
# ============================================================

# Current notebook: Milestone 2 / Improve_4.ipynb
model_df_path = Path("../Strategy_4/model_df.csv")
regime_path = Path("orderbook_with_kmeans_regime.csv")
peer_alpha_path = Path("../Strategy_5/peer_alpha_data.csv")

model_df = pd.read_csv(model_df_path)
regime_df = pd.read_csv(regime_path)
peer_alpha_df = pd.read_csv(peer_alpha_path)

print("model_df shape:", model_df.shape)
print("regime_df shape:", regime_df.shape)
print("peer_alpha_df shape:", peer_alpha_df.shape)


# ============================================================
# 2. Reconstruct stock column in model_df
# ============================================================

def infer_stock_from_dummies(df):
    """
    model_df has stock dummies:
        stock_GOOG
        stock_INTC
        stock_MSFT

    AMZN is the base case:
        if all stock dummies are 0, stock = AMZN
    """

    df = df.copy()

    df["stock"] = "AMZN"

    if "stock_GOOG" in df.columns:
        df.loc[df["stock_GOOG"] == 1, "stock"] = "GOOG"

    if "stock_INTC" in df.columns:
        df.loc[df["stock_INTC"] == 1, "stock"] = "INTC"

    if "stock_MSFT" in df.columns:
        df.loc[df["stock_MSFT"] == 1, "stock"] = "MSFT"

    return df


model_df = infer_stock_from_dummies(model_df)

print("\nStock distribution in model_df:")
print(model_df["stock"].value_counts())


# ============================================================
# 3. Merge K-Means regime labels
# ============================================================

regime_labels = regime_df[
    ["stock", "Time", "OrderID", "regime"]
].copy()

regime_labels = regime_labels.drop_duplicates(
    subset=["stock", "Time", "OrderID"],
    keep="first"
)

model_df = model_df.merge(
    regime_labels,
    on=["stock", "Time", "OrderID"],
    how="left"
)

print("\nAfter regime merge:", model_df.shape)
print("Missing regime:", model_df["regime"].isna().sum())
print("Missing regime pct:", model_df["regime"].isna().mean())
print(model_df["regime"].value_counts(dropna=False).sort_index())

# Drop rows without regime if any
model_df = model_df.dropna(subset=["regime"]).copy()
model_df["regime"] = model_df["regime"].astype(int)


# ============================================================
# 4. Merge peer alpha data
# ============================================================

peer_cols = [
    "stock",
    "Time",
    "OrderID",
    "peer_VWOF",
    "peer_MicroPrice",
    "peer_MicroMomentum",
    "peer_Spread",
    "peer_Spread_Limit",
    "peer_spread_safe",
    "peer_event_frac_in_min",
]

peer_cols = [c for c in peer_cols if c in peer_alpha_df.columns]

peer_alpha_df = peer_alpha_df[peer_cols].copy()

peer_alpha_df = peer_alpha_df.drop_duplicates(
    subset=["stock", "Time", "OrderID"],
    keep="first"
)

model_df = model_df.merge(
    peer_alpha_df,
    on=["stock", "Time", "OrderID"],
    how="left"
)

print("\nAfter peer alpha merge:", model_df.shape)

peer_alpha_cols = [
    "peer_VWOF",
    "peer_MicroPrice",
    "peer_MicroMomentum",
    "peer_Spread",
    "peer_Spread_Limit",
    "peer_spread_safe",
    "peer_event_frac_in_min",
]

peer_alpha_cols = [c for c in peer_alpha_cols if c in model_df.columns]

print("\nPeer alpha missing values:")
print(model_df[peer_alpha_cols].isna().sum())

# Fill missing peer alpha values if any
model_df[peer_alpha_cols] = (
    model_df[peer_alpha_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)


# ============================================================
# 5. Define your alphas + peer alphas
# ============================================================

your_alpha_cols = [
    "spread",
    "bid_liq_1",
    "ask_liq_1",
    "depth_value_1",
    "imbalance_1",
    "micro_minus_mid",
    "total_bid_size_5",
    "total_ask_size_5",
    "imbalance_5",
    "wap_bid_5",
    "wap_ask_5",
    "mid_chg1",
    "spread_chg1",
    "imbalance_1_chg1",
    "imbalance_5_chg1",
    "micro_minus_mid_chg1",
    "bid_liq_1_chg1",
    "ask_liq_1_chg1",
    "total_bid_size_5_chg1",
    "total_ask_size_5_chg1",
    "event_frac_in_min",
]

your_alpha_cols = [c for c in your_alpha_cols if c in model_df.columns]

peer_alpha_cols = [c for c in peer_alpha_cols if c in model_df.columns]

alpha_features_all = your_alpha_cols + peer_alpha_cols

# Remove duplicates if any
alpha_features_all = list(dict.fromkeys(alpha_features_all))

control_features = [
    "stock_GOOG",
    "stock_INTC",
    "stock_MSFT",
]

control_features = [c for c in control_features if c in model_df.columns]

# Convert bool controls to int
bool_cols = model_df.select_dtypes(include="bool").columns
model_df[bool_cols] = model_df[bool_cols].astype(int)

print("\nYour alpha count:", len(your_alpha_cols))
print("Peer alpha count:", len(peer_alpha_cols))
print("Total alpha count:", len(alpha_features_all))

print("\nYour alphas:")
print(your_alpha_cols)

print("\nPeer alphas:")
print(peer_alpha_cols)

print("\nControls:")
print(control_features)


# ============================================================
# 6. Robust alpha significance test by regime
# ============================================================

def test_alpha_significance_by_regime(
    df,
    target_col,
    alpha_features,
    control_features=None,
    min_obs=1000,
    hac_lags=5
):
    """
    For each regime, run:

        target_col ~ alpha_features + stock controls

    Since target is regret:
        coef < 0 = helpful, predicts lower regret
        coef > 0 = harmful, predicts higher regret

    Uses HAC robust standard errors.
    """

    if control_features is None:
        control_features = []

    results = []
    feature_cols = alpha_features + control_features

    for regime in sorted(df["regime"].dropna().unique()):

        sub = df[df["regime"] == regime].copy()

        needed_cols = [target_col] + feature_cols
        sub = sub[needed_cols].replace([np.inf, -np.inf], np.nan)

        for col in needed_cols:
            sub[col] = pd.to_numeric(sub[col], errors="coerce")

        sub = sub.dropna()

        if len(sub) < min_obs:
            print(f"Skipping regime {regime}: only {len(sub)} observations")
            continue

        y = sub[target_col].to_numpy(dtype=float)
        X_df = sub[feature_cols].astype(float)

        # Remove zero-variance columns within regime
        non_constant_cols = X_df.columns[X_df.std() > 0].tolist()
        X_df = X_df[non_constant_cols]

        X = sm.add_constant(X_df, has_constant="add").to_numpy(dtype=float)

        fitted = sm.OLS(y, X).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": hac_lags}
        )

        param_names = ["const"] + non_constant_cols
        params = pd.Series(fitted.params, index=param_names)
        tvalues = pd.Series(fitted.tvalues, index=param_names)
        pvalues = pd.Series(fitted.pvalues, index=param_names)

        for alpha in alpha_features:

            if alpha in params.index:
                coef = params[alpha]
                tval = tvalues[alpha]
                pval = pvalues[alpha]
                used = True
            else:
                coef = np.nan
                tval = np.nan
                pval = np.nan
                used = False

            if alpha in your_alpha_cols:
                alpha_source = "mine"
            elif alpha in peer_alpha_cols:
                alpha_source = "peer"
            else:
                alpha_source = "unknown"

            results.append({
                "target": target_col,
                "regime": int(regime),
                "alpha_source": alpha_source,
                "alpha": alpha,
                "coef": coef,
                "t_stat": tval,
                "p_value": pval,
                "significant_5pct": bool(pval < 0.05) if not pd.isna(pval) else False,
                "significant_1pct": bool(pval < 0.01) if not pd.isna(pval) else False,
                "helpful": bool((pval < 0.05) and (coef < 0)) if not pd.isna(pval) else False,
                "harmful": bool((pval < 0.05) and (coef > 0)) if not pd.isna(pval) else False,
                "n_obs": int(fitted.nobs),
                "r_squared": fitted.rsquared,
                "used_in_regression": used,
            })

    result_df = pd.DataFrame(results)

    if len(result_df) > 0:
        result_df = result_df.sort_values(
            ["target", "regime", "p_value"],
            na_position="last"
        )

    return result_df


# ============================================================
# 7. Run significance tests
# ============================================================

buy_sig = test_alpha_significance_by_regime(
    df=model_df,
    target_col="buy_regret",
    alpha_features=alpha_features_all,
    control_features=control_features,
    min_obs=1000,
    hac_lags=5
)

sell_sig = test_alpha_significance_by_regime(
    df=model_df,
    target_col="sell_regret",
    alpha_features=alpha_features_all,
    control_features=control_features,
    min_obs=1000,
    hac_lags=5
)

sig_results = pd.concat([buy_sig, sell_sig], ignore_index=True)

print("\nSignificance result head:")
display(sig_results.head(30))


# ============================================================
# 8. Extract helpful significant alphas only
# ============================================================

helpful_sig_alphas = sig_results[
    (sig_results["significant_5pct"] == True) &
    (sig_results["coef"] < 0)
].copy()

harmful_sig_alphas = sig_results[
    (sig_results["significant_5pct"] == True) &
    (sig_results["coef"] > 0)
].copy()

print("\nHelpful + significant alpha count:", len(helpful_sig_alphas))
print("Harmful + significant alpha count:", len(harmful_sig_alphas))

display(
    helpful_sig_alphas[
        ["target", "regime", "alpha_source", "alpha", "coef", "t_stat", "p_value", "n_obs", "r_squared"]
    ].sort_values(["target", "regime", "alpha_source", "p_value"])
)


# ============================================================
# 9. Summary: list helpful alphas by target/regime/source
# ============================================================

helpful_summary = (
    helpful_sig_alphas
    .groupby(["target", "regime", "alpha_source"])["alpha"]
    .apply(list)
    .reset_index()
)

print("\nHelpful alpha summary:")
display(helpful_summary)


# ============================================================
# 10. Pretty print
# ============================================================

for target in sorted(sig_results["target"].unique()):
    print("=" * 80)
    print(f"Target: {target}")
    print("=" * 80)

    for regime in sorted(sig_results["regime"].unique()):
        print(f"\nRegime {regime}:")

        sub = helpful_sig_alphas[
            (helpful_sig_alphas["target"] == target) &
            (helpful_sig_alphas["regime"] == regime)
        ].copy()

        if sub.empty:
            print("  No helpful significant alphas")
        else:
            for _, row in sub.sort_values(["alpha_source", "p_value"]).iterrows():
                print(
                    f"  [{row['alpha_source']}] {row['alpha']}: "
                    f"coef={row['coef']:.6g}, "
                    f"t={row['t_stat']:.3f}, "
                    f"p={row['p_value']:.4g}"
                )

model_df shape: (1149529, 72)
regime_df shape: (1149529, 62)
peer_alpha_df shape: (1149529, 10)

Stock distribution in model_df:
stock
MSFT    443425
INTC    430718
AMZN    175338
GOOG    100048
Name: count, dtype: int64

After regime merge: (1149529, 74)
Missing regime: 0
Missing regime pct: 0.0
regime
0    161739
1    763028
2     25000
3    195324
4      4438
Name: count, dtype: int64

After peer alpha merge: (1149529, 81)

Peer alpha missing values:
peer_VWOF                 0
peer_MicroPrice           0
peer_MicroMomentum        0
peer_Spread               0
peer_Spread_Limit         0
peer_spread_safe          0
peer_event_frac_in_min    0
dtype: int64

Your alpha count: 21
Peer alpha count: 7
Total alpha count: 28

Your alphas:
['spread', 'bid_liq_1', 'ask_liq_1', 'depth_value_1', 'imbalance_1', 'micro_minus_mid', 'total_bid_size_5', 'total_ask_size_5', 'imbalance_5', 'wap_bid_5', 'wap_ask_5', 'mid_chg1', 'spread_chg1', 'imbalance_1_chg1', 'imbalance_5_chg1', 'micro_minus_mid_ch

,target,regime,alpha_source,alpha,coef,t_stat,p_value,significant_5pct,significant_1pct,helpful,harmful,n_obs,r_squared,used_in_regression
0,buy_regret,0,peer,peer_spread_safe,1.747091e-02,5.219903e-03,0.995835,False,False,False,False,161739,0.375115,True
1,buy_regret,0,mine,wap_ask_5,-1.396326e-01,-2.462378e-03,0.998035,False,False,False,False,161739,0.375115,True
2,buy_regret,0,mine,total_bid_size_5,-3.758750e-07,-2.422165e-03,0.998067,False,False,False,False,161739,0.375115,True
3,buy_regret,0,mine,micro_minus_mid_chg1,1.256952e-01,2.064903e-03,0.998352,False,False,False,False,161739,0.375115,True
4,buy_regret,0,peer,peer_MicroPrice,6.649726e-01,1.828154e-03,0.998541,False,False,False,False,161739,0.375115,True
5,buy_regret,0,mine,wap_bid_5,-5.194525e-01,-1.527500e-03,0.998781,False,False,False,False,161739,0.375115,True
6,buy_regret,0,mine,total_ask_size_5,-1.780268e-06,-1.429766e-03,0.998859,False,False,False,False,161739,0.375115,True
7,buy_regret,0,mine,micro_minus_mid,-7.414978e-01,-1.287619e-03,0.998973,False,False,False,False,161739,0.375115,True
8,buy_regret,0,peer,peer_MicroMomentum,3.341025e-01,1.236025e-03,0.999014,False,False,False,False,161739,0.375115,True
9,buy_regret,0,peer,peer_Spread,-1.952204e-02,-9.252515e-04,0.999262,False,False,False,False,161739,0.375115,True



Helpful + significant alpha count: 33
Harmful + significant alpha count: 26


,target,regime,alpha_source,alpha,coef,t_stat,p_value,n_obs,r_squared
57,buy_regret,2,mine,wap_bid_5,-6.891794e-01,-9.784602,1.311096e-22,25000,0.563804
60,buy_regret,2,mine,wap_ask_5,-3.553898e-01,-7.334271,2.229321e-13,25000,0.563804
61,buy_regret,2,mine,micro_minus_mid,-8.704704e-01,-6.787461,1.141244e-11,25000,0.563804
64,buy_regret,2,mine,total_ask_size_5,-1.101736e-06,-5.640827,1.692356e-08,25000,0.563804
65,buy_regret,2,mine,ask_liq_1_chg1,-7.149544e-08,-3.345383,8.216889e-04,25000,0.563804
68,buy_regret,2,mine,total_bid_size_5_chg1,-5.663756e-07,-2.501686,1.236036e-02,25000,0.563804
69,buy_regret,2,mine,bid_liq_1_chg1,-2.389182e-08,-2.408469,1.601957e-02,25000,0.563804
58,buy_regret,2,peer,peer_VWOF,-5.593648e-02,-7.949385,1.874391e-15,25000,0.563804
71,buy_regret,2,peer,peer_Spread,-2.258127e-01,-2.186892,2.875040e-02,25000,0.563804
114,buy_regret,4,mine,wap_bid_5,-6.566298e-01,-8.394896,4.663087e-17,4438,0.614938



Helpful alpha summary:


,target,regime,alpha_source,alpha
0,buy_regret,2,mine,"[wap_bid_5, wap_ask_5, micro_minus_mid, total_..."
1,buy_regret,2,peer,"[peer_VWOF, peer_Spread]"
2,buy_regret,4,mine,"[wap_bid_5, micro_minus_mid, spread_chg1, ask_..."
3,buy_regret,4,peer,"[peer_Spread_Limit, peer_MicroMomentum, peer_V..."
4,sell_regret,2,mine,"[mid_chg1, total_ask_size_5, imbalance_5, ask_..."
5,sell_regret,4,mine,"[mid_chg1, total_ask_size_5, imbalance_5, even..."
6,sell_regret,4,peer,"[peer_Spread_Limit, peer_MicroPrice, peer_Spread]"


Target: buy_regret

Regime 0:
  No helpful significant alphas

Regime 1:
  No helpful significant alphas

Regime 2:
  [mine] wap_bid_5: coef=-0.689179, t=-9.785, p=1.311e-22
  [mine] wap_ask_5: coef=-0.35539, t=-7.334, p=2.229e-13
  [mine] micro_minus_mid: coef=-0.87047, t=-6.787, p=1.141e-11
  [mine] total_ask_size_5: coef=-1.10174e-06, t=-5.641, p=1.692e-08
  [mine] ask_liq_1_chg1: coef=-7.14954e-08, t=-3.345, p=0.0008217
  [mine] total_bid_size_5_chg1: coef=-5.66376e-07, t=-2.502, p=0.01236
  [mine] bid_liq_1_chg1: coef=-2.38918e-08, t=-2.408, p=0.01602
  [peer] peer_VWOF: coef=-0.0559365, t=-7.949, p=1.874e-15
  [peer] peer_Spread: coef=-0.225813, t=-2.187, p=0.02875

Regime 3:
  No helpful significant alphas

Regime 4:
  [mine] wap_bid_5: coef=-0.65663, t=-8.395, p=4.663e-17
  [mine] micro_minus_mid: coef=-1.25603, t=-6.722, p=1.792e-11
  [mine] spread_chg1: coef=-0.178328, t=-3.908, p=9.314e-05
  [mine] ask_liq_1_chg1: coef=-3.54687e-07, t=-3.351, p=0.000806
  [mine] imbalance_5_

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score


# ============================================================
# 11. Build regime-specific alpha map from helpful significant alphas
# ============================================================

def build_regime_alpha_map(helpful_sig_alphas):
    """
    Build mapping:

        target -> regime -> alpha list

    Only uses helpful + significant alphas.
    Regime 0/1/3 will naturally be excluded if they have no helpful alphas.
    """

    regime_alpha_map = {}

    for (target, regime), g in helpful_sig_alphas.groupby(["target", "regime"]):
        alphas = g["alpha"].dropna().unique().tolist()

        if target not in regime_alpha_map:
            regime_alpha_map[target] = {}

        regime_alpha_map[target][int(regime)] = alphas

    return regime_alpha_map


REGIME_ALPHA_MAP = build_regime_alpha_map(helpful_sig_alphas)

print("Regime alpha map:")
for target, regime_dict in REGIME_ALPHA_MAP.items():
    print(f"\nTarget: {target}")
    for regime, alphas in regime_dict.items():
        print(f"  Regime {regime}: {alphas}")

Regime alpha map:

Target: buy_regret
  Regime 2: ['wap_bid_5', 'peer_VWOF', 'wap_ask_5', 'micro_minus_mid', 'total_ask_size_5', 'ask_liq_1_chg1', 'total_bid_size_5_chg1', 'bid_liq_1_chg1', 'peer_Spread']
  Regime 4: ['wap_bid_5', 'micro_minus_mid', 'peer_Spread_Limit', 'spread_chg1', 'peer_MicroMomentum', 'ask_liq_1_chg1', 'peer_VWOF', 'imbalance_5_chg1', 'bid_liq_1', 'wap_ask_5']

Target: sell_regret
  Regime 2: ['mid_chg1', 'total_ask_size_5', 'imbalance_5', 'ask_liq_1_chg1', 'event_frac_in_min', 'imbalance_1_chg1']
  Regime 4: ['mid_chg1', 'total_ask_size_5', 'peer_Spread_Limit', 'imbalance_5', 'event_frac_in_min', 'peer_MicroPrice', 'micro_minus_mid_chg1', 'peer_Spread']


In [17]:
# ============================================================
# 12. Prepare dataframe for regime-switching Ridge
# ============================================================

def prepare_df_for_regime_strategy(df):
    df = df.copy()

    # Convert stock dummy bools to int if needed
    bool_cols = df.select_dtypes(include="bool").columns
    df[bool_cols] = df[bool_cols].astype(int)

    # Make sure ts and minute exist
    if "ts" not in df.columns:
        df["ts"] = pd.to_datetime(df["Time"], format="%H:%M:%S.%f", errors="coerce")

    if "minute" not in df.columns:
        df["minute"] = df["ts"].dt.floor("min")

    df = df.dropna(subset=["ts", "minute", "regime"]).copy()
    df["regime"] = df["regime"].astype(int)

    df = df.sort_values(["stock", "ts"]).reset_index(drop=True)

    return df


model_df = prepare_df_for_regime_strategy(model_df)

In [18]:
# ============================================================
# 13. Train/test split by stock-minute
# ============================================================

def train_test_split_by_stock_minute(df, train_frac=0.7):
    """
    Chronological split within each stock.
    This avoids mixing future minutes into training.
    """

    train_parts = []
    test_parts = []

    for stock, g in df.groupby("stock"):
        g = g.sort_values("ts").copy()

        unique_minutes = np.array(sorted(g["minute"].unique()))
        split_idx = int(len(unique_minutes) * train_frac)

        train_minutes = unique_minutes[:split_idx]
        test_minutes = unique_minutes[split_idx:]

        train_parts.append(g[g["minute"].isin(train_minutes)])
        test_parts.append(g[g["minute"].isin(test_minutes)])

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, test_df


train_df, test_df = train_test_split_by_stock_minute(model_df, train_frac=0.7)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain regime distribution:")
print(train_df["regime"].value_counts().sort_index())

print("\nTest regime distribution:")
print(test_df["regime"].value_counts().sort_index())

Train shape: (883579, 81)
Test shape: (265950, 81)

Train regime distribution:
regime
0    137628
1    565994
2     18877
3    157099
4      3981
Name: count, dtype: int64

Test regime distribution:
regime
0     24111
1    197034
2      6123
3     38225
4       457
Name: count, dtype: int64


In [20]:
# ============================================================
# 14. Fit one Ridge model per target-regime
# Fixed for older sklearn versions
# ============================================================

def fit_regime_ridge_models(
    train_df,
    regime_alpha_map,
    control_features,
    ridge_alpha=10.0,
    min_obs=1000
):
    """
    Fit one Ridge model for each target-regime pair.

    This is one model per target-regime across all stocks,
    with stock dummy controls included.
    """

    ridge_models = {}
    summaries = []

    for target_col, regime_dict in regime_alpha_map.items():

        for regime, alpha_cols in regime_dict.items():

            alpha_cols = [c for c in alpha_cols if c in train_df.columns]
            controls = [c for c in control_features if c in train_df.columns]

            feature_cols = alpha_cols + controls
            feature_cols = list(dict.fromkeys(feature_cols))

            sub = train_df[train_df["regime"] == regime].copy()

            needed_cols = [target_col] + feature_cols
            sub = sub[needed_cols].replace([np.inf, -np.inf], np.nan)

            for col in needed_cols:
                sub[col] = pd.to_numeric(sub[col], errors="coerce")

            sub = sub.dropna()

            if len(sub) < min_obs:
                print(f"Skipping {target_col}, Regime {regime}: only {len(sub)} rows")
                continue

            X = sub[feature_cols]
            y = sub[target_col]

            model = Pipeline([
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=ridge_alpha))
            ])

            model.fit(X, y)

            pred_train = model.predict(X)

            train_rmse = np.sqrt(mean_squared_error(y, pred_train))
            train_r2 = r2_score(y, pred_train)

            ridge_models[(target_col, regime)] = {
                "model": model,
                "alpha_cols": alpha_cols,
                "control_cols": controls,
                "feature_cols": feature_cols,
            }

            summaries.append({
                "target": target_col,
                "regime": regime,
                "n_obs": len(sub),
                "n_alphas": len(alpha_cols),
                "n_controls": len(controls),
                "train_rmse": train_rmse,
                "train_r2": train_r2,
                "alpha_cols": alpha_cols,
                "control_cols": controls,
            })

    return ridge_models, pd.DataFrame(summaries)


ridge_models, ridge_train_summary = fit_regime_ridge_models(
    train_df=train_df,
    regime_alpha_map=REGIME_ALPHA_MAP,
    control_features=control_features,
    ridge_alpha=10.0,
    min_obs=1000
)

display(ridge_train_summary)

,target,regime,n_obs,n_alphas,n_controls,train_rmse,train_r2,alpha_cols,control_cols
0,buy_regret,2,18877,9,3,0.168149,0.469831,"[wap_bid_5, peer_VWOF, wap_ask_5, micro_minus_...","[stock_GOOG, stock_INTC, stock_MSFT]"
1,buy_regret,4,3981,10,3,0.395353,0.310396,"[wap_bid_5, micro_minus_mid, peer_Spread_Limit...","[stock_GOOG, stock_INTC, stock_MSFT]"
2,sell_regret,2,18877,6,3,0.127609,0.203339,"[mid_chg1, total_ask_size_5, imbalance_5, ask_...","[stock_GOOG, stock_INTC, stock_MSFT]"
3,sell_regret,4,3981,8,3,0.232641,0.215397,"[mid_chg1, total_ask_size_5, peer_Spread_Limit...","[stock_GOOG, stock_INTC, stock_MSFT]"


In [21]:
# ============================================================
# 15. Evaluate Ridge models on test data
# ============================================================

def evaluate_regime_ridge_models(test_df, ridge_models):
    results = []

    for (target_col, regime), obj in ridge_models.items():

        model = obj["model"]
        feature_cols = obj["feature_cols"]

        sub = test_df[test_df["regime"] == regime].copy()

        needed_cols = [target_col] + feature_cols
        sub = sub[needed_cols].replace([np.inf, -np.inf], np.nan)

        for col in needed_cols:
            sub[col] = pd.to_numeric(sub[col], errors="coerce")

        sub = sub.dropna()

        if len(sub) == 0:
            continue

        X = sub[feature_cols]
        y = sub[target_col]

        pred = model.predict(X)

        test_rmse = np.sqrt(mean_squared_error(y, pred))
        test_r2 = r2_score(y, pred)

        results.append({
            "target": target_col,
            "regime": regime,
            "n_obs": len(sub),
            "test_rmse": test_rmse,
            "test_r2": test_r2,
        })

    return pd.DataFrame(results)


ridge_test_summary = evaluate_regime_ridge_models(test_df, ridge_models)

display(ridge_test_summary)

,target,regime,n_obs,test_rmse,test_r2
0,buy_regret,2,6123,0.094080,0.101749
1,buy_regret,4,457,0.174765,-0.868436
2,sell_regret,2,6123,0.088652,0.163445
3,sell_regret,4,457,0.126766,-0.014352


In [23]:
# ============================================================
# 16. Add predicted regret to test data
# ============================================================

def add_regime_ridge_predictions(df, ridge_models):
    df = df.copy()

    df["pred_buy_regret"] = np.nan
    df["pred_sell_regret"] = np.nan

    for (target_col, regime), obj in ridge_models.items():

        model = obj["model"]
        feature_cols = obj["feature_cols"]

        mask = df["regime"] == regime

        sub = df.loc[mask, feature_cols].copy()
        sub = sub.replace([np.inf, -np.inf], np.nan)

        for col in feature_cols:
            sub[col] = pd.to_numeric(sub[col], errors="coerce")

        valid_idx = sub.dropna().index

        if len(valid_idx) == 0:
            continue

        pred = model.predict(df.loc[valid_idx, feature_cols])

        if target_col == "buy_regret":
            df.loc[valid_idx, "pred_buy_regret"] = pred
        elif target_col == "sell_regret":
            df.loc[valid_idx, "pred_sell_regret"] = pred

    return df


train_pred_df = add_regime_ridge_predictions(train_df, ridge_models)
test_pred_df = add_regime_ridge_predictions(test_df, ridge_models)

print(test_pred_df[["stock", "Time", "minute", "regime", "pred_buy_regret", "pred_sell_regret"]].head())

  stock          Time               minute  regime  pred_buy_regret  \
0  AMZN  12:39:05.651  1900-01-01 12:39:00       0              NaN   
1  AMZN  12:39:09.397  1900-01-01 12:39:00       0              NaN   
2  AMZN  12:39:09.397  1900-01-01 12:39:00       2         0.057042   
3  AMZN  12:39:09.397  1900-01-01 12:39:00       0              NaN   
4  AMZN  12:39:09.397  1900-01-01 12:39:00       0              NaN   

   pred_sell_regret  
0               NaN  
1               NaN  
2          0.151043  
3               NaN  
4               NaN  


In [24]:
# ============================================================
# 17. Fit thresholds from training predictions
# ============================================================

def fit_prediction_thresholds(train_pred_df, q=0.20):
    thresholds = {}

    for target_col, pred_col in [
        ("buy_regret", "pred_buy_regret"),
        ("sell_regret", "pred_sell_regret"),
    ]:
        for regime in sorted(train_pred_df["regime"].unique()):

            sub = train_pred_df[
                (train_pred_df["regime"] == regime) &
                (train_pred_df[pred_col].notna())
            ]

            if len(sub) == 0:
                continue

            thresholds[(target_col, int(regime))] = sub[pred_col].quantile(q)

    return thresholds


thresholds = fit_prediction_thresholds(train_pred_df, q=0.20)

print("Thresholds:")
print(thresholds)

Thresholds:
{('buy_regret', 2): np.float64(0.021804302436810977), ('buy_regret', 4): np.float64(0.1641529632515159), ('sell_regret', 2): np.float64(0.040370945118505926), ('sell_regret', 4): np.float64(0.10725989257797758)}


In [25]:
# ============================================================
# 18. Execute regime-switching Ridge strategy
# ============================================================

def execute_regime_switching_ridge_strategy(
    df,
    side,
    thresholds,
    twap_regimes=(0, 1, 3),
    ridge_regimes=(2, 4),
    fallback="twap"
):
    df = df.copy()
    df = df.sort_values(["stock", "minute", "ts"])

    side = side.upper()

    if side == "BUY":
        exec_col = "AskPrice_1"
        pred_col = "pred_buy_regret"
        target_col = "buy_regret"
    elif side == "SELL":
        exec_col = "BidPrice_1"
        pred_col = "pred_sell_regret"
        target_col = "sell_regret"
    else:
        raise ValueError("side must be BUY or SELL")

    records = []

    for (stock, minute), g in df.groupby(["stock", "minute"]):

        g = g.sort_values("ts").copy()

        twap_row = g.iloc[0]
        twap_price = twap_row[exec_col]

        chosen_row = None
        chosen_reason = None

        first_regime = int(twap_row["regime"])

        # Regime 0/1/3: TWAP at first observation of minute
        if first_regime in twap_regimes:
            chosen_row = twap_row
            chosen_reason = f"twap_regime_{first_regime}"

        else:
            # Regime 2/4: Ridge trigger
            for _, row in g.iterrows():

                regime = int(row["regime"])

                if regime not in ridge_regimes:
                    continue

                key = (target_col, regime)

                if key not in thresholds:
                    continue

                pred_regret = row[pred_col]

                if pd.isna(pred_regret):
                    continue

                if pred_regret <= thresholds[key]:
                    chosen_row = row
                    chosen_reason = f"ridge_trigger_regime_{regime}"
                    break

            if chosen_row is None:
                if fallback == "twap":
                    chosen_row = twap_row
                    chosen_reason = "fallback_twap_no_trigger"
                elif fallback == "last":
                    chosen_row = g.iloc[-1]
                    chosen_reason = "fallback_last_no_trigger"
                else:
                    raise ValueError("fallback must be 'twap' or 'last'")

        adaptive_price = chosen_row[exec_col]

        if side == "BUY":
            improvement = twap_price - adaptive_price
        else:
            improvement = adaptive_price - twap_price

        records.append({
            "stock": stock,
            "minute": minute,
            "side": side,
            "twap_price": twap_price,
            "adaptive_price": adaptive_price,
            "improvement": improvement,
            "chosen_time": chosen_row["Time"],
            "chosen_regime": int(chosen_row["regime"]),
            "chosen_reason": chosen_reason,
            "pred_regret": chosen_row[pred_col],
        })

    return pd.DataFrame(records)

In [26]:
# ============================================================
# 19. Generate exec_results
# ============================================================

buy_exec = execute_regime_switching_ridge_strategy(
    df=test_pred_df,
    side="BUY",
    thresholds=thresholds,
    twap_regimes=(0, 1, 3),
    ridge_regimes=(2, 4),
    fallback="twap"
)

sell_exec = execute_regime_switching_ridge_strategy(
    df=test_pred_df,
    side="SELL",
    thresholds=thresholds,
    twap_regimes=(0, 1, 3),
    ridge_regimes=(2, 4),
    fallback="twap"
)

exec_results = pd.concat([buy_exec, sell_exec], ignore_index=True)

print(exec_results.shape)
display(exec_results.head())

(648, 10)


,stock,minute,side,twap_price,adaptive_price,improvement,chosen_time,chosen_regime,chosen_reason,pred_regret
0,AMZN,1900-01-01 12:39:00,BUY,222.45,222.45,0.0,12:39:05.651,0,twap_regime_0,NaN
1,AMZN,1900-01-01 12:40:00,BUY,222.54,222.54,0.0,12:40:01.032,0,twap_regime_0,NaN
2,AMZN,1900-01-01 12:41:00,BUY,222.54,222.54,0.0,12:41:00.098,0,twap_regime_0,NaN
3,AMZN,1900-01-01 12:42:00,BUY,222.43,222.43,0.0,12:42:00.510,0,twap_regime_0,NaN
4,AMZN,1900-01-01 12:43:00,BUY,222.62,222.62,0.0,12:43:00.045,3,twap_regime_3,NaN


In [27]:
# ============================================================
# 20. Export result in Strategy 4 format
# ============================================================

regime_ridge_buy_mean = (
    exec_results[exec_results["side"] == "BUY"]
    .groupby("stock", as_index=False)["improvement"]
    .mean()
    .rename(columns={"improvement": "average_improvement"})
)

regime_ridge_sell_mean = (
    exec_results[exec_results["side"] == "SELL"]
    .groupby("stock", as_index=False)["improvement"]
    .mean()
    .rename(columns={"improvement": "average_improvement"})
)

display(regime_ridge_buy_mean)
display(regime_ridge_sell_mean)

regime_ridge_buy_mean.to_csv("regime_ridge_buy_execution.csv", index=False)
regime_ridge_sell_mean.to_csv("regime_ridge_sell_execution.csv", index=False)

exec_results.to_csv("regime_ridge_execution_detail.csv", index=False)

print("Saved:")
print("regime_ridge_buy_execution.csv")
print("regime_ridge_sell_execution.csv")
print("regime_ridge_execution_detail.csv")

,stock,average_improvement
0,AMZN,0.000123
1,GOOG,0.005556
2,INTC,0.000000
3,MSFT,0.000000


,stock,average_improvement
0,AMZN,0.000988
1,GOOG,0.008025
2,INTC,0.000000
3,MSFT,0.000000


Saved:
regime_ridge_buy_execution.csv
regime_ridge_sell_execution.csv
regime_ridge_execution_detail.csv


In [28]:
# ============================================================
# 23. Non-regime Ridge benchmark
# Use all helpful + significant alphas, ignore regime
# ============================================================

def build_global_alpha_map(helpful_sig_alphas):
    """
    Build one alpha list per target using all helpful significant alphas,
    ignoring regime.
    """

    global_alpha_map = {}

    for target, g in helpful_sig_alphas.groupby("target"):
        global_alpha_map[target] = g["alpha"].dropna().unique().tolist()

    return global_alpha_map


GLOBAL_ALPHA_MAP = build_global_alpha_map(helpful_sig_alphas)

print("Global alpha map:")
for target, alphas in GLOBAL_ALPHA_MAP.items():
    print(f"{target}: {alphas}")


def fit_global_ridge_models(
    train_df,
    global_alpha_map,
    control_features,
    ridge_alpha=10.0,
    min_obs=1000
):
    """
    Fit one global Ridge model for buy_regret and one for sell_regret.
    Ignore regime, but include stock dummy controls.
    """

    global_models = {}
    summaries = []

    for target_col, alpha_cols in global_alpha_map.items():

        alpha_cols = [c for c in alpha_cols if c in train_df.columns]
        controls = [c for c in control_features if c in train_df.columns]

        feature_cols = list(dict.fromkeys(alpha_cols + controls))

        sub = train_df.copy()

        needed_cols = [target_col] + feature_cols
        sub = sub[needed_cols].replace([np.inf, -np.inf], np.nan)

        for col in needed_cols:
            sub[col] = pd.to_numeric(sub[col], errors="coerce")

        sub = sub.dropna()

        if len(sub) < min_obs:
            print(f"Skipping global {target_col}: only {len(sub)} rows")
            continue

        X = sub[feature_cols]
        y = sub[target_col]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=ridge_alpha))
        ])

        model.fit(X, y)
        pred_train = model.predict(X)

        global_models[target_col] = {
            "model": model,
            "alpha_cols": alpha_cols,
            "control_cols": controls,
            "feature_cols": feature_cols,
        }

        summaries.append({
            "target": target_col,
            "n_obs": len(sub),
            "n_alphas": len(alpha_cols),
            "n_controls": len(controls),
            "train_rmse": np.sqrt(mean_squared_error(y, pred_train)),
            "train_r2": r2_score(y, pred_train),
            "alpha_cols": alpha_cols,
            "control_cols": controls,
        })

    return global_models, pd.DataFrame(summaries)


global_models, global_train_summary = fit_global_ridge_models(
    train_df=train_df,
    global_alpha_map=GLOBAL_ALPHA_MAP,
    control_features=control_features,
    ridge_alpha=10.0,
    min_obs=1000
)

display(global_train_summary)

Global alpha map:
buy_regret: ['wap_bid_5', 'peer_VWOF', 'wap_ask_5', 'micro_minus_mid', 'total_ask_size_5', 'ask_liq_1_chg1', 'total_bid_size_5_chg1', 'bid_liq_1_chg1', 'peer_Spread', 'peer_Spread_Limit', 'spread_chg1', 'peer_MicroMomentum', 'imbalance_5_chg1', 'bid_liq_1']
sell_regret: ['mid_chg1', 'total_ask_size_5', 'imbalance_5', 'ask_liq_1_chg1', 'event_frac_in_min', 'imbalance_1_chg1', 'peer_Spread_Limit', 'peer_MicroPrice', 'micro_minus_mid_chg1', 'peer_Spread']


,target,n_obs,n_alphas,n_controls,train_rmse,train_r2,alpha_cols,control_cols
0,buy_regret,883579,14,3,0.082949,0.410860,"[wap_bid_5, peer_VWOF, wap_ask_5, micro_minus_...","[stock_GOOG, stock_INTC, stock_MSFT]"
1,sell_regret,883579,10,3,0.067160,0.354081,"[mid_chg1, total_ask_size_5, imbalance_5, ask_...","[stock_GOOG, stock_INTC, stock_MSFT]"


In [29]:
# ============================================================
# 24. Add global Ridge predictions
# ============================================================

def add_global_ridge_predictions(df, global_models):
    df = df.copy()

    df["global_pred_buy_regret"] = np.nan
    df["global_pred_sell_regret"] = np.nan

    for target_col, obj in global_models.items():

        model = obj["model"]
        feature_cols = obj["feature_cols"]

        sub = df[feature_cols].copy()
        sub = sub.replace([np.inf, -np.inf], np.nan)

        for col in feature_cols:
            sub[col] = pd.to_numeric(sub[col], errors="coerce")

        valid_idx = sub.dropna().index

        if len(valid_idx) == 0:
            continue

        pred = model.predict(df.loc[valid_idx, feature_cols])

        if target_col == "buy_regret":
            df.loc[valid_idx, "global_pred_buy_regret"] = pred
        elif target_col == "sell_regret":
            df.loc[valid_idx, "global_pred_sell_regret"] = pred

    return df


global_train_pred_df = add_global_ridge_predictions(train_df, global_models)
global_test_pred_df = add_global_ridge_predictions(test_df, global_models)

In [30]:
# ============================================================
# 25. Fit global thresholds
# ============================================================

def fit_global_thresholds(global_train_pred_df, q=0.20):
    """
    Lower predicted regret is better.
    One threshold for buy and one threshold for sell.
    """

    thresholds = {}

    pred_map = {
        "buy_regret": "global_pred_buy_regret",
        "sell_regret": "global_pred_sell_regret",
    }

    for target_col, pred_col in pred_map.items():
        sub = global_train_pred_df[global_train_pred_df[pred_col].notna()]

        if len(sub) == 0:
            continue

        thresholds[target_col] = sub[pred_col].quantile(q)

    return thresholds


global_thresholds = fit_global_thresholds(global_train_pred_df, q=0.20)

print("Global thresholds:")
print(global_thresholds)

Global thresholds:
{'buy_regret': np.float64(0.00614207776948778), 'sell_regret': np.float64(-0.0015072816476800696)}


In [31]:
# ============================================================
# 26. Execute non-regime Ridge strategy
# ============================================================

def execute_global_ridge_strategy(
    df,
    side,
    global_thresholds,
    fallback="twap"
):
    """
    Ignore regime.

    For every stock-minute:
        scan from beginning of the minute
        execute at first row where predicted regret <= threshold

    If no trigger:
        fallback to TWAP, first row of the minute.
    """

    df = df.copy()
    df = df.sort_values(["stock", "minute", "ts"])

    side = side.upper()

    if side == "BUY":
        exec_col = "AskPrice_1"
        pred_col = "global_pred_buy_regret"
        target_col = "buy_regret"
    elif side == "SELL":
        exec_col = "BidPrice_1"
        pred_col = "global_pred_sell_regret"
        target_col = "sell_regret"
    else:
        raise ValueError("side must be BUY or SELL")

    if target_col not in global_thresholds:
        raise ValueError(f"No global threshold found for {target_col}")

    threshold = global_thresholds[target_col]

    records = []

    for (stock, minute), g in df.groupby(["stock", "minute"]):

        g = g.sort_values("ts").copy()

        twap_row = g.iloc[0]
        twap_price = twap_row[exec_col]

        chosen_row = None
        chosen_reason = None

        for _, row in g.iterrows():
            pred_regret = row[pred_col]

            if pd.isna(pred_regret):
                continue

            if pred_regret <= threshold:
                chosen_row = row
                chosen_reason = "global_ridge_trigger"
                break

        if chosen_row is None:
            if fallback == "twap":
                chosen_row = twap_row
                chosen_reason = "fallback_twap_no_trigger"
            elif fallback == "last":
                chosen_row = g.iloc[-1]
                chosen_reason = "fallback_last_no_trigger"
            else:
                raise ValueError("fallback must be 'twap' or 'last'")

        adaptive_price = chosen_row[exec_col]

        if side == "BUY":
            improvement = twap_price - adaptive_price
        else:
            improvement = adaptive_price - twap_price

        records.append({
            "stock": stock,
            "minute": minute,
            "side": side,
            "twap_price": twap_price,
            "adaptive_price": adaptive_price,
            "improvement": improvement,
            "chosen_time": chosen_row["Time"],
            "chosen_regime": int(chosen_row["regime"]),
            "chosen_reason": chosen_reason,
            "pred_regret": chosen_row[pred_col],
        })

    return pd.DataFrame(records)


global_buy_exec = execute_global_ridge_strategy(
    df=global_test_pred_df,
    side="BUY",
    global_thresholds=global_thresholds,
    fallback="twap"
)

global_sell_exec = execute_global_ridge_strategy(
    df=global_test_pred_df,
    side="SELL",
    global_thresholds=global_thresholds,
    fallback="twap"
)

global_exec_results = pd.concat([global_buy_exec, global_sell_exec], ignore_index=True)

display(global_exec_results.head())

,stock,minute,side,twap_price,adaptive_price,improvement,chosen_time,chosen_regime,chosen_reason,pred_regret
0,AMZN,1900-01-01 12:39:00,BUY,222.45,222.44,0.01,12:39:09.398,0,global_ridge_trigger,0.002886
1,AMZN,1900-01-01 12:40:00,BUY,222.54,222.54,0.00,12:40:01.032,0,fallback_twap_no_trigger,0.075568
2,AMZN,1900-01-01 12:41:00,BUY,222.54,222.54,0.00,12:41:00.098,0,fallback_twap_no_trigger,0.092274
3,AMZN,1900-01-01 12:42:00,BUY,222.43,222.43,0.00,12:42:00.510,0,fallback_twap_no_trigger,0.057214
4,AMZN,1900-01-01 12:43:00,BUY,222.62,222.57,0.05,12:43:54.278,2,global_ridge_trigger,0.002734


In [32]:
# ============================================================
# 27. Export non-regime Ridge result in same format
# ============================================================

global_ridge_buy_mean = (
    global_exec_results[global_exec_results["side"] == "BUY"]
    .groupby("stock", as_index=False)["improvement"]
    .mean()
    .rename(columns={"improvement": "average_improvement"})
)

global_ridge_sell_mean = (
    global_exec_results[global_exec_results["side"] == "SELL"]
    .groupby("stock", as_index=False)["improvement"]
    .mean()
    .rename(columns={"improvement": "average_improvement"})
)

display(global_ridge_buy_mean)
display(global_ridge_sell_mean)

global_ridge_buy_mean.to_csv("global_ridge_buy_execution.csv", index=False)
global_ridge_sell_mean.to_csv("global_ridge_sell_execution.csv", index=False)
global_exec_results.to_csv("global_ridge_execution_detail.csv", index=False)

print("Saved:")
print("global_ridge_buy_execution.csv")
print("global_ridge_sell_execution.csv")
print("global_ridge_execution_detail.csv")

,stock,average_improvement
0,AMZN,0.022716
1,GOOG,0.024568
2,INTC,0.000123
3,MSFT,0.000494


,stock,average_improvement
0,AMZN,0.002716
1,GOOG,0.000864
2,INTC,0.000494
3,MSFT,-0.000247


Saved:
global_ridge_buy_execution.csv
global_ridge_sell_execution.csv
global_ridge_execution_detail.csv
